# Эксперименты с маскированием

1. *Runtime → Change runtime type → GPU*.
2. Запустите ячейки по порядку. После `git pull` с новым кодом: *Runtime → Restart session* и снова с первой ячейки.

In [ ]:
# Установка: клонирует репозиторий (или обновляет, если он уже есть) и ставит зависимости + pyg-lib
import os

os.chdir("/content")
if os.path.exists("science-article-gnn"):
    !git -C science-article-gnn pull
else:
    !git clone https://github.com/Nok1sh/science-article-gnn.git
%cd /content/science-article-gnn
!python install.py


In [ ]:
# (Необязательно) Google Drive, чтобы результаты пережили отключение Colab
from google.colab import drive

drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/gnn_results"
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
# Быстрая проверка: одна модель на исходных данных, оценка на исходных и замаскированных
import warnings

warnings.filterwarnings("ignore")
import masks
from data import load_data
from training import train

original = load_data("rel-f1", "driver-position")
noisy = load_data("rel-f1", "driver-position", mask=masks.noise(1.0))

model = train(original, epochs=15, verbose=False)
print("best epoch:", model.best_epoch, model.best_val_metrics)
print(original.name, model.evaluate(original))
print(noisy.name, model.evaluate(noisy))


## Перебор масок

Для каждой задачи и сида обучается эталонная модель на исходных данных, затем для каждой маски считаются два протокола:

- `masked_trained` — модель обучена и проверена на замаскированных данных (реалистичный случай: доступна только замаскированная база);
- `original_trained` — эталонная модель проверена на замаскированных данных (насколько модель опирается на точные значения). Для `generalize(..., "range")` не считается: меняется тип колонок.

Два сида нужны, чтобы видеть разброс от случайности: разница между масками меньше этого разброса — шум.
Ориентировочно ~1 минута на обучение на GPU, всего ~60 обучений.

In [ ]:
from experiments import run_sweep

RESULTS_DIR = globals().get("RESULTS_DIR", "/content")
CSV_PATH = f"{RESULTS_DIR}/sweep.csv"

tasks = [
    ("rel-f1", "driver-position"),      # регрессия, NMAE (меньше — лучше)
    ("rel-f1", "driver-dnf"),           # классификация, ROC-AUC (больше — лучше)
    ("rel-f1", "driver-top3"),          # классификация, ROC-AUC
    ("rel-f1", "qualifying-position"),  # autocomplete-регрессия, NMAE
]

mask_list = [
    masks.noise(1.0),                    # сильный шум в числах
    masks.generalize(5, "midpoint"),     # 500 -> середина диапазона (число)
    masks.generalize(3, "range"),        # 500 -> "400-600" (категория)
    masks.suppress(0.5),                 # половина значений признаков удалена
    masks.suppress(1.0),                 # все значения признаков удалены: остаются только связи и время
    masks.drop_links(0.5),               # половина внешних ключей (рёбер графа) удалена
    masks.compose(                       # комбинированная маска
        masks.generalize(3, "range"),
        masks.suppress(0.5),
        masks.drop_links(0.3),
    ),
]

df = run_sweep(tasks, mask_list, seeds=(42, 43), csv_path=CSV_PATH, epochs=15)
print("saved to", CSV_PATH)


In [2]:
# Сводная таблица по test: среднее по сидам (± разброс) и изменение относительно исходных данных
import pandas as pd

df = pd.read_csv("sweep.csv")
test = df[df["split"] == "test"].copy()
test["score"] = test.get("nmae", pd.Series(index=test.index, dtype=float)).combine_first(
    test.get("roc_auc", pd.Series(index=test.index, dtype=float))
)
test["mask"] = pd.Categorical(test["mask"], categories=list(dict.fromkeys(test["mask"])), ordered=True)

summary = (
    test.groupby(["task", "protocol", "mask"], observed=True)["score"]
    .agg(mean="mean", std="std")
    .reset_index()
)
baseline = summary[summary["mask"] == "original"].set_index("task")["mean"]
summary["delta_vs_original"] = summary["mean"] - summary["task"].map(baseline)

for task, part in summary.groupby("task"):
    print()
    print(f"=== {task}  ({'NMAE, меньше — лучше' if 'position' in task else 'ROC-AUC, больше — лучше'})")
    display(part.pivot(index="mask", columns="protocol", values=["mean", "delta_vs_original"]).round(4))



=== rel-f1/driver-dnf  (ROC-AUC, больше — лучше)


mean  \
protocol                                           masked_trained   
mask                                                                
original                                                      NaN   
noise(1.0)                                                 0.7087   
generalize(5,midpoint,quantile)                            0.7086   
generalize(3,range,quantile)                               0.7137   
suppress(0.5)                                              0.6972   
suppress(1.0)                                              0.6912   
drop_links(0.5)                                            0.7269   
generalize(3,range,quantile)+suppress(0.5)+drop...         0.6763   

                                                                     \
protocol                                           original_trained   
mask                                                                  
original                                                     0.7344   
noise(1.0)                                                   0.7299   
generalize(5,midpoint,quantile)                              0.7059   
generalize(3,range,quantile)                                    NaN   
suppress(0.5)                                                0.6699   
suppress(1.0)                                                0.6561   
drop_links(0.5)                                              0.7436   
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN   

                                                   delta_vs_original  \
protocol                                              masked_trained   
mask                                                                   
original                                                         NaN   
noise(1.0)                                                   -0.0256   
generalize(5,midpoint,quantile)                              -0.0257   
generalize(3,range,quantile)                                 -0.0207   
suppress(0.5)                                                -0.0372   
suppress(1.0)                                                -0.0431   
drop_links(0.5)                                              -0.0074   
generalize(3,range,quantile)+suppress(0.5)+drop...           -0.0580   

                                                                     
protocol                                           original_trained  
mask                                                                 
original                                                     0.0000  
noise(1.0)                                                  -0.0045  
generalize(5,midpoint,quantile)                             -0.0285  
generalize(3,range,quantile)                                    NaN  
suppress(0.5)                                               -0.0645  
suppress(1.0)                                               -0.0782  
drop_links(0.5)                                              0.0092  
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN


=== rel-f1/driver-position  (NMAE, меньше — лучше)


mean  \
protocol                                           masked_trained   
mask                                                                
original                                                      NaN   
noise(1.0)                                                 0.5654   
generalize(5,midpoint,quantile)                            0.5702   
generalize(3,range,quantile)                               0.5960   
suppress(0.5)                                              0.6122   
suppress(1.0)                                              0.6111   
drop_links(0.5)                                            0.5942   
generalize(3,range,quantile)+suppress(0.5)+drop...         0.6349   

                                                                     \
protocol                                           original_trained   
mask                                                                  
original                                                     0.5894   
noise(1.0)                                                   0.5905   
generalize(5,midpoint,quantile)                              0.5977   
generalize(3,range,quantile)                                    NaN   
suppress(0.5)                                                0.5942   
suppress(1.0)                                                0.6266   
drop_links(0.5)                                              0.5894   
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN   

                                                   delta_vs_original  \
protocol                                              masked_trained   
mask                                                                   
original                                                         NaN   
noise(1.0)                                                   -0.0239   
generalize(5,midpoint,quantile)                              -0.0191   
generalize(3,range,quantile)                                  0.0066   
suppress(0.5)                                                 0.0228   
suppress(1.0)                                                 0.0217   
drop_links(0.5)                                               0.0048   
generalize(3,range,quantile)+suppress(0.5)+drop...            0.0455   

                                                                     
protocol                                           original_trained  
mask                                                                 
original                                                     0.0000  
noise(1.0)                                                   0.0012  
generalize(5,midpoint,quantile)                              0.0083  
generalize(3,range,quantile)                                    NaN  
suppress(0.5)                                                0.0048  
suppress(1.0)                                                0.0373  
drop_links(0.5)                                              0.0001  
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN


=== rel-f1/driver-top3  (ROC-AUC, больше — лучше)


mean  \
protocol                                           masked_trained   
mask                                                                
original                                                      NaN   
noise(1.0)                                                 0.8105   
generalize(5,midpoint,quantile)                            0.7991   
generalize(3,range,quantile)                               0.7354   
suppress(0.5)                                              0.7387   
suppress(1.0)                                              0.7215   
drop_links(0.5)                                            0.7985   
generalize(3,range,quantile)+suppress(0.5)+drop...         0.7832   

                                                                     \
protocol                                           original_trained   
mask                                                                  
original                                                     0.7890   
noise(1.0)                                                   0.7794   
generalize(5,midpoint,quantile)                              0.7957   
generalize(3,range,quantile)                                    NaN   
suppress(0.5)                                                0.7707   
suppress(1.0)                                                0.6451   
drop_links(0.5)                                              0.7904   
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN   

                                                   delta_vs_original  \
protocol                                              masked_trained   
mask                                                                   
original                                                         NaN   
noise(1.0)                                                    0.0215   
generalize(5,midpoint,quantile)                               0.0100   
generalize(3,range,quantile)                                 -0.0536   
suppress(0.5)                                                -0.0503   
suppress(1.0)                                                -0.0676   
drop_links(0.5)                                               0.0095   
generalize(3,range,quantile)+suppress(0.5)+drop...           -0.0058   

                                                                     
protocol                                           original_trained  
mask                                                                 
original                                                     0.0000  
noise(1.0)                                                  -0.0096  
generalize(5,midpoint,quantile)                              0.0067  
generalize(3,range,quantile)                                    NaN  
suppress(0.5)                                               -0.0184  
suppress(1.0)                                               -0.1440  
drop_links(0.5)                                              0.0014  
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN


=== rel-f1/qualifying-position  (NMAE, меньше — лучше)


mean  \
protocol                                           masked_trained   
mask                                                                
original                                                      NaN   
noise(1.0)                                                 0.7124   
generalize(5,midpoint,quantile)                            0.6686   
generalize(3,range,quantile)                               0.7588   
suppress(0.5)                                              0.8115   
suppress(1.0)                                              0.7793   
drop_links(0.5)                                            0.6987   
generalize(3,range,quantile)+suppress(0.5)+drop...         0.7220   

                                                                     \
protocol                                           original_trained   
mask                                                                  
original                                                     0.6592   
noise(1.0)                                                   0.6765   
generalize(5,midpoint,quantile)                              0.6748   
generalize(3,range,quantile)                                    NaN   
suppress(0.5)                                                0.6739   
suppress(1.0)                                                0.8327   
drop_links(0.5)                                              0.7549   
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN   

                                                   delta_vs_original  \
protocol                                              masked_trained   
mask                                                                   
original                                                         NaN   
noise(1.0)                                                    0.0531   
generalize(5,midpoint,quantile)                               0.0094   
generalize(3,range,quantile)                                  0.0996   
suppress(0.5)                                                 0.1523   
suppress(1.0)                                                 0.1200   
drop_links(0.5)                                               0.0395   
generalize(3,range,quantile)+suppress(0.5)+drop...            0.0628   

                                                                     
protocol                                           original_trained  
mask                                                                 
original                                                     0.0000  
noise(1.0)                                                   0.0173  
generalize(5,midpoint,quantile)                              0.0156  
generalize(3,range,quantile)                                    NaN  
suppress(0.5)                                                0.0146  
suppress(1.0)                                                0.1734  
drop_links(0.5)                                              0.0957  
generalize(3,range,quantile)+suppress(0.5)+drop...              NaN